# Difference in Differences

Difference in Differences (DiD) is a statistical technique used to estimate the causal
effect of a specific intervention or treatment (such as a law change, a new policy or
a marketing campaign) by comparing the changes in outcomes over time between a
"treatment" group and a "control" group.

As we will see later in this lecture, when DiD studies have more than two periods of
time per unit of observation, we use fixed effects.

---

Imports

In [ ]:
%%capture
!pip install linearmodels

In [ ]:
from linearmodels.panel.model import PanelOLS
import numpy as np
import pandas as pd

## Fixed Effects

Fixed Effects (FEs) are used to control for unobserved variables that are constant
within a specific group. Whether you are looking at people, states, or birth cohorts,
FEs allow you to ignore "level differences" and focus entirely on how changes in your
independent variable ($X$) lead to changes in your outcome ($Y$). Instead of a single
average starting point, each cohort gets its own baseline.

### Panel Data

Suppose we can't observe someone's IQ, but we want to know how years of
experience (YOE) affects their wages.

$$
    Y_{it} = \beta_0 + \beta_1 IQ_i + \beta_2 YOE_{it} + \varepsilon_{it}
$$

If we only have two periods, we could subtract $Y_{it-1}$ from $Y_{it}$. Since
IQ doesn't change over time, the term $(\beta_1 IQ_i - \beta_1 IQ_i)$ cancels
out, leaving us with a "clean" estimate of $\beta_2$.

$$
    \Delta Y_{it} = \beta_2 \Delta YOE_{it} + \Delta \varepsilon_{it}
$$

In this setup, $\beta_1 IQ_i$ is essentially a "custom" starting point for
each person. Rather than assuming everyone starts at the same $\beta_0$, we
give every individual $i$ their own intercept, $\alpha_i$, which captures
all their unique, constant traits (like IQ).

$$
    Y_{it} = \alpha_i + \beta_2 YOE_{it} + \varepsilon_{it}
$$

While we could just add a dummy variable for every person, inverting a matrix
with thousands of columns is computationally expensive. Instead, we can
achieve the same result by "demeaning" the data. In other words, we can
subtract the average value of each variable for each person.

$$
    Y_{it} - \bar{Y}_i = \beta_2 (YOE_{it} - \overline{YOE}_i)
    + (\varepsilon_{it} - \bar{\varepsilon}_i)
$$

In [ ]:
# Init a list to hold the data
data = []

# Four individuals (entities)
for i in range(1, 5):
    # Two periods per individual
    for t in range(1, 3):
        data.append({'i': i, 't': t})

# Create the DataFrame from list
df = pd.DataFrame(data)

# Make up people's time to run a mile
df['x'] = [9.8, 9.2, 9.9, 9.6, 13.8, 11.3, 13.5, 12.7]

# Make up people's BMI
df['y'] = 25 + 0.1 * df['i'] - 0.4 * df['t'] - 0.5 * df['x']  # No noise

# Display df in raw form
display(df)

### One Way FEs

Let's set up the data to fit the model
$Y_{it} = \alpha_i + \beta X_{it} + \varepsilon_{it}$.

This is how we need to format the data to pass it to `PanelOLS`.

In [ ]:
# Create copy
df_entity = df.copy()

# Set index
df_entity.set_index('i', inplace=True)

# View result
display(df_entity)

We can also set time units as fixed effects to model
$Y_{it} = \lambda_t + \beta X_{it} + \varepsilon_{it}$.

In [ ]:
# Create copy
df_time = df.copy()

# Set index
df_time = df_time.set_index('t').sort_index()

# View result
display(df_time)

### Two-Way Fixed Effects (TWFE)

Let's set up the data to fit the model
$Y_{it} = \alpha_i + \lambda_t + \beta X_{it} + \varepsilon_{it}$

In [ ]:
# Create copy
df_twfe = df.copy()

# Set index
df_twfe.set_index(['i', 't'], inplace=True)

# View result
display(df_twfe)

## Difference in Differences

### Two Periods and Two Groups

Suppose we have two time periods and two groups. We'll enconde the first
time period as $t = 0$ and the second one as $t = 1$. Likewise, we will use
$D = 0$ for control units and $D = 1$ for treatment units.

$$
    E(Y_i|t, D_i) = \beta_0 + \beta_1 D_i + \beta_2 t + \beta_3 t D_i
$$

- $E(Y | t_0, Control) = \beta_0$
- $E(Y | t_1, Control) = \beta_0 + \beta_2$
- $E(Y | t_0, Treatment) = \beta_0 + \beta_1$
- $E(Y | t_1, Treatment) = \beta_0 + \beta_1 + \beta_2 + \beta_3$

> **How do you get $\beta_3$ on its own?**

Simple, you take the difference of the following differences:

1. $\Delta_1 = E(Y | t_1, Control) - E(Y | t_0, Control) = \beta_2$
2. $
    \Delta_2 = E(Y | t_1, Treatment) - E(Y | t_0, Treatment)
    = \beta_2 + \beta_3
$

$$ \implies \beta_3 = \Delta_2 - \Delta_1 $$

$\beta_3$ captures the treatment effect under the assumption that if the
treatment group had never been treated, it would have followed the same
trend as the control group (i.e., both groups would follow *parallel trends*)

> This is why $\beta_3$ is known as the *Difference in Differences*
estimator.

#### Minimum Wages and Employment (Card and Krueger, 1994)

In 1992, the state of New Jersey raised the minimum wage, while the neighboring
state of Pennsylvania did not. Card and Krueger use this natural experiment to
study the effect that increasing the minimum wage has on employment.

To do so, they use fast-food restaurants in both states to see how the number
of full-time employees changed between 1991 (before the policy took place) and
1992 (after the policy kicked in).

##### No Controls

We will start off with a $2 \times 2$ design and no controls.

In [ ]:
# Load Data
URL = 'https://raw.githubusercontent.com/ArturoSbr/econometrics-ii-2025/refs/heads/main/assignments/did/data/card-krueger.csv'
df = pd.read_csv(URL)[[
    'i', 't', 'empft', 'state', 'psoda', 'pfry', 'pentree', 'nmgrs', 'status_1'
]]

# Only keep restaurants that answered the second interview

# Declare treatment column
df['treat'] = None

# Set indexes

# Declare model
model = PanelOLS()

# Fit model
result = None

# View results
print(result)

##### Adding Controls

First off, we should only include controls that vary over time. Otherwise,
they are already absorbed by each unit's fixed effect ($\alpha_i$). Beyond
that, there's a major risk of adding "bad controls." For example, if raising
the minimum wage causes restaurants to hike prices, adding `pfry` would
soak up some of the policy's effect and bias our estimate for `treat`.

In practice, we solve this by controlling for covariates **before** the
policy took place. We record the characteristics of restaurants in 1991 to
capture a baseline that is unaffected by the subsequent wage hike.
Mathematically, instead of using $X_{it}$, we use $X_i$ (the 1991 values).

Since $X_i$ is constant over time, it would normally be absorbed by the
entity fixed effects. To prevent these *snapshots* from being dropped, we
interact them with time ($t$). This allows the model to account for
different trends based on initial characteristics without introducing
endogeneity.

$$
    E(Y_{it} | t, D_i, X_i) = \beta_0 + \beta_1 D_i + \beta_2 t +
    \beta_3 (t \cdot D_i) + \gamma (t \cdot X_i)
$$

In [ ]:
# Get snapshot (covariates **before** the policy)
controls = ['psoda', 'pfry', 'pentree', 'nmgrs']
snapshot = df.sort_index(level=['i', 't']).groupby(level='i')[controls].first()

# Join snapshot back to df

# Interact snapshot columns with time

# Init model
model_X = PanelOLS()

# Fit model
result_X = None

# View results
print(result_X)

### Multiple Periods and Heterogeneous Treatment Effect

An Event Study is a version of DiD that models heterogeneous treatment
effects across time. $t$ is used to model time-level FEs, while $k$ is
a new variable that models the number of periods until a unit gets
treated. Mathematically, for a unit treated in period $t = \tau$, its
new time variable is calculated using $k = t - \tau$.

Periods where $k \lt 0$ are called *leads* (think of the periods *leading*
up to the policy), and periods where $k \ge 0$ are called *lags*. We normally
set $k = -1$ as the baseline because it represents the last period before the
treatment began, though any other $k$ could be set as the baseline.

$$
    Y_{it} = \alpha_i + \lambda_t + \sum_{k \neq -1} \delta_k D_{it}^k
    + \varepsilon_{it}
$$

* $\alpha_i$: Unit fixed effects (soaks up person/entity baselines).
* $\lambda_t$: Time fixed effects (soaks up shocks common to everyone).
* $k$: "Event time" (periods relative to treatment).
* $D_{it}^k$: A dummy that is 1 if unit $i$ is exactly $k$ periods away
from their treatment date at time $t$.
* $\delta_k$: The dynamic treatment effect at period $k$.

A **strong assumption** made by this model is that the treatment group remains
treated after the first time they receive the treatment. In other words, units
cannot switch their treatment status back to control after they've been treated.

#### Testing Parallel Trends (Anticipated Treatment Effects)

Notice that this model allows us to test for pre-treatment effects! Ideally,
we want no anticipated effects, because in theory, the treatment effects should
begin **after** a unit becomes affected by a policy, **not before**!

To test for anticipated treatment effects, we must perform an $F$-test to check
that $\delta_l = 0 \space \forall \space l \lt 1$. Performing individual
$t$-tests is not enough because the hypothesis must test the significance of all
leading periods simultaneously.

#### A Word of Caution

Event Studies were introduced in 1999, and for about 22 years, researchers
used them to study natural experiments where adoption was staggered (e.g.,
unit $i$ is treated in 2005, and unit $j$ in 2007).

However, recent work—most notably by Goodman-Bacon (2021)—showed that
standard TWFE regressions can produce biased results in this setting. The
problem is the "Forbidden Comparison": the model often uses **already-treated**
units as controls for **newly-treated** units (e.g., a unit treated in 2005
is used as a counterfactual for a unit treated in 2007).

If the effect of the treatment changes over time (e.g., if the policy gets
stronger the longer it's in place), using an already-treated unit as a
benchmark will attenuate the estimated effect on units treated later. In
other words, you end up subtracting the *change* in the early-adopters'
outcome from the *change* in the late-adopters' outcome.

This can lead to "negative weights," where a positive treatment effect
actually shows up as negative in your results. Essentially, the OLS
estimator gets confused by time-varying treatment effects when calculating
the average. Today, we use "clean" estimators (like Callaway & Sant'Anna)
that explicitly forbid using already-treated units as controls.

In summary, if the treatment is rolled out gradually (i.e., adoption is
staggered), **DO NOT USE EVENT STUDIES**! Use a more modern method, such as
the one proposed by [Callaway & Stant'Anna (2021)](
    https://www.sciencedirect.com/science/article/abs/pii/S0304407620303948
).

In [ ]:
# Read data
URL_CS = 'https://raw.githubusercontent.com/ArturoSbr/econometrics-ii-2025/refs/heads/main/assignments/did/data/callaway-santanna.csv'
df = None

# Only keep rows that are never treated or treated in 2006

# Declare event-time variable (k)

# Turn k into dummies

# Set indexes

# Declare list of regressors
exog = []

# Decalre model
model_es = PanelOLS()

# Fit model
results_es = None

# View results
print(results_es)